# Feature Importance — Credit Risk Model

This notebook analyzes which features drive model predictions:

- XGBoost built-in importance (gain, cover, weight)
- SHAP summary plot (top 20 features)
- SHAP dependence plots for top 3 features
- Interpretation of credit risk drivers

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import shap
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder
from sklearn.base import BaseEstimator, TransformerMixin
from xgboost import XGBClassifier

plt.rcParams.update({"figure.dpi": 120, "font.size": 11})
RANDOM_STATE = 42

## 1. Train Model (same pipeline as `train_register.py`)

In [ ]:
df = pd.read_csv("../data/complete_feature_dataset.csv")

FEATURES = [
    "EXT_SOURCE_3", "EXT_SOURCE_2", "EXT_SOURCE_1", "DAYS_BIRTH",
    "AMT_GOODS_PRICE", "AMT_CREDIT", "AMT_ANNUITY", "DAYS_EMPLOYED",
    "CODE_GENDER", "BUREAU_DEBT_TO_CREDIT_RATIO", "BUREAU_ACTIVE_CREDIT_SUM",
    "NAME_EDUCATION_TYPE", "POS_MEAN_CONTRACT_LENGTH", "PREV_ANNUITY_MEAN",
    "PREV_GOODS_TO_CREDIT_RATIO", "NAME_FAMILY_STATUS", "POS_LATEST_MONTH",
    "ORGANIZATION_TYPE", "BUREAU_AMT_MAX_OVERDUE_EVER", "POS_TOTAL_MONTHS_OBSERVED",
    "AMT_INCOME_TOTAL", "PREV_REFUSAL_RATE", "NAME_INCOME_TYPE", "CNT_CHILDREN",
]
CAT_COLS = ["CODE_GENDER", "NAME_EDUCATION_TYPE", "NAME_FAMILY_STATUS",
            "ORGANIZATION_TYPE", "NAME_INCOME_TYPE"]
NUM_COLS = [c for c in FEATURES if c not in CAT_COLS]

X = df[FEATURES]; y = df["TARGET"].astype(int)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

class ClipQuantiles(BaseEstimator, TransformerMixin):
    def __init__(self, qlow=0.001, qhigh=0.999):
        self.qlow, self.qhigh = qlow, qhigh
    def fit(self, X, y=None):
        Xdf = pd.DataFrame(X)
        self.lower_ = Xdf.quantile(self.qlow, interpolation="nearest")
        self.upper_ = Xdf.quantile(self.qhigh, interpolation="nearest")
        return self
    def transform(self, X):
        return pd.DataFrame(X).clip(self.lower_, self.upper_, axis=1).values

prev_ratio_cols = [c for c in NUM_COLS if c.startswith("PREV_") and ("RATIO" in c or "RATE" in c)]
prev_other_cols = [c for c in NUM_COLS if c.startswith("PREV_") and c not in prev_ratio_cols]
other_num_cols  = [c for c in NUM_COLS if not c.startswith("PREV_")]

transformers = []
if other_num_cols:
    transformers.append(("num_other", Pipeline([
        ("clip", ClipQuantiles()), ("impute", SimpleImputer(strategy="median"))
    ]), other_num_cols))
if prev_ratio_cols:
    transformers.append(("num_prev_ratio", Pipeline([
        ("clip", ClipQuantiles()), ("impute", SimpleImputer(strategy="median"))
    ]), prev_ratio_cols))
if prev_other_cols:
    transformers.append(("num_prev_other", Pipeline([
        ("clip", ClipQuantiles()), ("impute", SimpleImputer(strategy="constant", fill_value=0))
    ]), prev_other_cols))
if CAT_COLS:
    transformers.append(("cat", Pipeline([
        ("impute", SimpleImputer(strategy="most_frequent")),
        ("ord", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ]), CAT_COLS))

pre = ColumnTransformer(transformers, remainder="drop")
clf = XGBClassifier(
    n_estimators=300, max_depth=4, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8, eval_metric="logloss",
    n_jobs=-1, random_state=RANDOM_STATE,
)
pipe = Pipeline([("pre", pre), ("clf", clf)])
pipe.fit(X_train, y_train)

# Get transformed feature names in order
transformed_feature_names = (
    other_num_cols + prev_ratio_cols + prev_other_cols + CAT_COLS
)
# Preprocessed test set for SHAP
X_test_transformed = pipe.named_steps["pre"].transform(X_test)
X_test_df = pd.DataFrame(X_test_transformed, columns=transformed_feature_names)
print(f"Trained. Features after preprocessing: {X_test_df.shape[1]}")

## 2. XGBoost Built-in Feature Importance (Gain, Cover, Weight)

In [ ]:
xgb_model = pipe.named_steps["clf"]

fig, axes = plt.subplots(1, 3, figsize=(18, 7))
for ax, imp_type in zip(axes, ["weight", "gain", "cover"]):
    imp = xgb_model.get_booster().get_score(importance_type=imp_type)
    # Map f0..fN back to feature names
    imp_named = {transformed_feature_names[int(k[1:])]: v for k, v in imp.items()
                 if k.startswith("f") and int(k[1:]) < len(transformed_feature_names)}
    imp_series = pd.Series(imp_named).sort_values()
    imp_series.tail(20).plot.barh(ax=ax)
    ax.set_title(f"Top 20 by {imp_type}")
    ax.set_xlabel(imp_type.capitalize())

plt.suptitle("XGBoost Built-in Feature Importance", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

## 3. SHAP Summary Plot (Top 20 Features)

In [ ]:
# Use a sample for SHAP (full test set can be slow)
sample_size = min(5000, len(X_test_df))
X_shap = X_test_df.sample(n=sample_size, random_state=RANDOM_STATE)

explainer = shap.TreeExplainer(xgb_model)
shap_values = explainer.shap_values(X_shap)

shap.summary_plot(shap_values, X_shap, max_display=20, show=False)
plt.title("SHAP Summary — Top 20 Features")
plt.tight_layout()
plt.show()

## 4. SHAP Dependence Plots — Top 3 Features

In [ ]:
# Identify top 3 features by mean absolute SHAP value
mean_abs_shap = np.abs(shap_values).mean(axis=0)
top3_idx = np.argsort(mean_abs_shap)[-3:][::-1]
top3_features = [transformed_feature_names[i] for i in top3_idx]

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, feat in zip(axes, top3_features):
    shap.dependence_plot(feat, shap_values, X_shap, ax=ax, show=False)
    ax.set_title(f"SHAP Dependence: {feat}")

plt.tight_layout()
plt.show()
print(f"Top 3 features by mean |SHAP|: {top3_features}")

## 5. Credit Risk Driver Interpretation

**Expected top drivers (based on domain knowledge and prior EDA):**

- **EXT_SOURCE_1/2/3** — External credit scoring agency scores. These are the strongest predictors because they aggregate a borrower's credit history from external bureaus. Higher values indicate lower risk.

- **DAYS_BIRTH** — Age of the applicant (negative days from application date). Older applicants tend to have lower default rates — more financial stability, longer credit history.

- **DAYS_EMPLOYED** — Employment duration. Longer employment correlates with income stability and lower default risk. The feature is negative for currently employed applicants.

- **AMT_CREDIT / AMT_ANNUITY / AMT_GOODS_PRICE** — Loan amount features. Higher credit amounts relative to income increase default risk. The ratio between these amounts captures affordability.

- **BUREAU_DEBT_TO_CREDIT_RATIO** — Existing debt burden from credit bureau records. High debt-to-credit ratios indicate financial stress and predict defaults.

- **PREV_REFUSAL_RATE** — Rate of previous application refusals. A history of being refused credit is a strong signal of elevated risk.

**Categorical features** (CODE_GENDER, NAME_EDUCATION_TYPE, etc.) contribute moderate but meaningful predictive power after ordinal encoding. Their SHAP values should be interpreted cautiously — ordinal encoding imposes an artificial ordering.